# SFT v2 — zayıf açıklama ayıklama + karma talimat verisi

v1'de model yalnızca TUS sorusu çözmeyi öğrendi ve başka talimatları takip edemez hale geldi (catastrophic forgetting). v2 iki şey yapıyor: zayıf açıklamaları ayıklıyor ve eğitim setine %6 oranında arama sorgusu üretme görevi karıştırıyor.

**Sonuç:** TUS 70,9% → 71,8%; talimat takibi bozuk → 309/316 (%98).

**Uyarı:** v2 RAG altında v1'den kötü çalışıyor (85,1% vs 88,6%, p = 0,007). `entegrasyon/README.md`'deki rol ayrımına bak.

## 1 kurulum

`hucreler/v2/1_kurulum.py`

In [ ]:
%%capture
!pip install -q unsloth

## 2 veri hazirla

`hucreler/v2/2_veri_hazirla.py`

In [ ]:
import os, re, json, glob, zipfile, random, unicodedata, torch
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

MODEL_NAME = "Qwen/Qwen3.5-4B"
ZIP_PATH   = "/content/drive/MyDrive/tus_data/TUS_Dataset_Final-20261002T154401Z-1-001.zip"
HEDEF, SEED = "/content/tus_veri", 42
KARMA_ORAN = 0.13          # egitim setinin ~%13'u talimat ornegi
MAX_SEQ_LEN = 2048

if not glob.glob(f"{HEDEF}/**/egitim.jsonl", recursive=True):
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(HEDEF)
DATA_DIR = str(Path(glob.glob(f"{HEDEF}/**/egitim.jsonl", recursive=True)[0]).parent)

CEV  = re.compile(r"^<think>\n(.*)\n</think>\n\nDoğru cevap: ([A-E])$", re.S)
SIK  = re.compile(r"^([A-E])\) (.+)$", re.M)
def nrm(s):
    s = unicodedata.normalize("NFD", s.lower()).translate(str.maketrans("çğıöşü", "cgiosu"))
    return re.sub(r"[^a-z0-9 ]", " ", s)

ham = [json.loads(l) for l in open(f"{DATA_DIR}/egitim.jsonl", encoding="utf-8")]
guclu, zayif = [], []
for r in ham:
    m = CEV.match(r["messages"][1]["content"])
    if not m: continue
    acik, harf = m.group(1), m.group(2)
    if len(acik.strip()) < 40: continue
    sk = dict(SIK.findall(r["messages"][0]["content"]))
    if list(sk) not in (list("ABCDE"), list("ABCD")) or harf not in sk: continue
    kel = [w for w in nrm(sk[harf]).split() if len(w) >= 5]
    (zayif if (kel and not any(w in nrm(acik) for w in kel)) else guclu).append(
        {"question": r["messages"][0]["content"], "letter": harf, "aciklama": acik,
         "sik": sk[harf], "group": (r.get("meta") or {}).get("group", "?")})
print(f"toplam {len(ham)} -> guclu {len(guclu)} | ayiklanan zayif {len(zayif)}")

def cevap_once(r):
    return {"gorev": "tus", "question": r["question"],
            "answer": f"Doğru cevap: {r['letter']}) {r['sik']}\n\nAçıklama: {r['aciklama'][:3000]}"}
tus_rows = [cevap_once(r) for r in guclu]

from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME, max_seq_length=MAX_SEQ_LEN, dtype=torch.bfloat16, load_in_4bit=False)
tok = getattr(tokenizer, "tokenizer", tokenizer)
if tok.pad_token is None: tok.pad_token = tok.eos_token
FastLanguageModel.for_inference(model)

def prompt(sis, kul):
    return tok.apply_chat_template([{"role":"system","content":sis},{"role":"user","content":kul}],
                                   tokenize=False, add_generation_prompt=True, enable_thinking=False)

@torch.no_grad()
def uret(prompts, mnt=64, bs=16, etiket=""):
    tok.padding_side = "left"; outs = []
    for i in range(0, len(prompts), bs):
        b = prompts[i:i+bs]
        enc = tok(b, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        g = model.generate(**enc, max_new_tokens=mnt, do_sample=False, pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(g[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"  {etiket}{min(i+bs,len(prompts))}/{len(prompts)}", end="\r")
    print(); return outs

SIS_SORGU = ("Sen bir tıp kütüphanecisisin. Sana bir TUS sorusu verilecek. Soruyu CEVAPLAMA. "
             "Yalnızca bu soruyu çözmek için ders kitabında aranması gereken tıbbi anahtar "
             "kelimeleri tek satırda, virgülle ayırarak yaz.")
SIS_TERIM = ("Sen bir tıp uzmanısın. Sana bir tıbbi terim verilecek. "
             "Terimi iki cümleyle, sade Türkçeyle açıkla. Soru sorma, liste yapma.")

random.Random(SEED).shuffle(guclu)
n_her = int(len(tus_rows) * KARMA_ORAN / 2)
print(f"\nkarma veri uretiliyor: {n_her} sorgu + {n_her} terim")

s_kaynak = guclu[:n_her]
s_out = uret([prompt(SIS_SORGU, r["question"]) for r in s_kaynak], 48, etiket="[sorgu] ")
def temiz(t):
    s = re.sub(r"<think>.*?</think>", "", t or "", flags=re.S).strip().split("\n")[0]
    return re.sub(r"^(arama|sorgu|anahtar kelimeler)\s*[:\-]\s*", "", s.strip(' "\'`'), flags=re.I).strip()
sorgu_rows = [{"gorev": "sorgu", "sistem": SIS_SORGU, "question": r["question"], "answer": temiz(o)}
              for r, o in zip(s_kaynak, s_out)
              if 5 <= len(temiz(o)) <= 200 and not re.match(r"^[A-E][\)\.]", temiz(o))]

TERIM = re.compile(r"\b[A-ZÇĞİÖŞÜ][a-zçğıöşü]{5,}(?:\s+[a-zçğıöşü]{4,})?\b")
havuz = []
for r in guclu[n_her:n_her*4]:
    t = TERIM.findall(r["aciklama"][:400])
    if t: havuz.append(t[0])
havuz = list(dict.fromkeys(havuz))[:n_her]
t_out = uret([prompt(SIS_TERIM, f"{t} nedir?") for t in havuz], 120, etiket="[terim] ")
terim_rows = [{"gorev": "terim", "sistem": SIS_TERIM, "question": f"{t} nedir?",
               "answer": re.sub(r"<think>.*?</think>", "", o, flags=re.S).strip()}
              for t, o in zip(havuz, t_out) if 40 <= len(o.strip()) <= 700]

print(f"\nkarma: sorgu {len(sorgu_rows)} | terim {len(terim_rows)}")
print("ornek sorgu :", sorgu_rows[0]["answer"][:90])
print("ornek terim :", terim_rows[0]["question"], "->", terim_rows[0]["answer"][:90])

egitim = tus_rows + sorgu_rows + terim_rows
random.Random(SEED).shuffle(egitim)
nv = max(1, int(len(egitim) * 0.02))
val_rows, train_rows = egitim[:nv], egitim[nv:]
print(f"\nTOPLAM egitim {len(train_rows)} | val {len(val_rows)}")
print(f"  tus {len(tus_rows)} (%{100*len(tus_rows)/len(egitim):.0f}) | "
      f"sorgu {len(sorgu_rows)} | terim {len(terim_rows)}")

test_rows = [{"question": r["soru"], "letter": r["dogru_cevap"],
              "group": (r.get("meta") or {}).get("group", "?")}
             for r in (json.loads(l) for l in open(f"{DATA_DIR}/test_cevap.jsonl", encoding="utf-8"))]
print(f"test {len(test_rows)}")

## 2b terim temizle

`hucreler/v2/2b_terim_temizle.py`

In [ ]:
import re

EK = re.compile(r"(?:da|de|ta|te|dan|den|tan|ten|ın|in|un|ün|nın|nin|nun|nün|"
                r"ya|ye|yi|yı|yu|yü|la|le|ile|lar|ler|sı|si|su|sü|nda|nde)$", re.I)

def saglam_terim(q):
    t = q[:-7].strip() if q.endswith(" nedir?") else q
    if not (3 <= len(t) <= 45): return False
    if " " in t and len(t.split()) > 3: return False
    son = t.split()[-1]
    if EK.search(son) and len(son) > 6: return False      # cekim eki tasiyor
    if re.search(r"[0-9,;:()\[\]]", t): return False
    return True

once = len(terim_rows)
atilan = [r for r in terim_rows if not saglam_terim(r["question"])]
terim_rows = [r for r in terim_rows if saglam_terim(r["question"])]
print(f"terim: {once} -> {len(terim_rows)} (atilan {len(atilan)})")
print("\natilan ornekler:")
for r in atilan[:6]: print("  -", r["question"])
print("\nkalan ornekler:")
for r in terim_rows[:6]: print("  +", r["question"])

import random
egitim = tus_rows + sorgu_rows + terim_rows
random.Random(SEED).shuffle(egitim)
nv = max(1, int(len(egitim) * 0.02))
val_rows, train_rows = egitim[:nv], egitim[nv:]
print(f"\nTOPLAM egitim {len(train_rows)} | val {len(val_rows)}")
print(f"  tus {len(tus_rows)} (%{100*len(tus_rows)/len(egitim):.0f}) | "
      f"sorgu {len(sorgu_rows)} | terim {len(terim_rows)} "
      f"(karma %{100*(len(sorgu_rows)+len(terim_rows))/len(egitim):.0f})")

## 2c terim yeniden

`hucreler/v2/2c_terim_yeniden.py`

In [ ]:
import re, random

KOTU = re.compile(r"^(hiç|hepsi|yukarıdaki|aşağıdaki|diğer|tümü|bunların|yalnız|sadece|I{1,3}\b)", re.I)
def temiz_terim(s):
    t = re.sub(r"\s*\(.*?\)", "", s).strip(" .,;:")
    t = re.sub(r"\s+", " ", t)
    return t

havuz = []
for r in guclu:
    t = temiz_terim(r["sik"])
    if 4 <= len(t) <= 48 and len(t.split()) <= 4 and not KOTU.match(t) \
       and not re.search(r"[0-9]{3,}|%", t):
        havuz.append(t)
havuz = list(dict.fromkeys(havuz))
random.Random(SEED).shuffle(havuz)
havuz = havuz[:1400]
print(f"sik metninden terim havuzu: {len(havuz)}")
print("ornekler:", havuz[:8])

t_out = uret([prompt(SIS_TERIM, f"{t} nedir?") for t in havuz], 120, bs=16, etiket="[terim] ")
terim_rows = [{"gorev": "terim", "sistem": SIS_TERIM, "question": f"{t} nedir?",
               "answer": re.sub(r"<think>.*?</think>", "", o, flags=re.S).strip()}
              for t, o in zip(havuz, t_out) if 40 <= len(o.strip()) <= 700]
print(f"\nterim ornegi: {len(terim_rows)}")
for r in terim_rows[:3]:
    print(f"  {r['question']}")
    print(f"    -> {r['answer'][:110]}")

egitim = tus_rows + sorgu_rows + terim_rows
random.Random(SEED).shuffle(egitim)
nv = max(1, int(len(egitim) * 0.02))
val_rows, train_rows = egitim[:nv], egitim[nv:]
print(f"\nTOPLAM egitim {len(train_rows)} | val {len(val_rows)}")
print(f"  tus {len(tus_rows)} | sorgu {len(sorgu_rows)} | terim {len(terim_rows)} "
      f"(karma %{100*(len(sorgu_rows)+len(terim_rows))/len(egitim):.0f})")

## 2d terim cikar

`hucreler/v2/2d_terim_cikar.py`

In [ ]:
import random
egitim = tus_rows + sorgu_rows
random.Random(SEED).shuffle(egitim)
nv = max(1, int(len(egitim) * 0.02))
val_rows, train_rows = egitim[:nv], egitim[nv:]
print(f"TOPLAM egitim {len(train_rows)} | val {len(val_rows)}")
print(f"  tus {len(tus_rows)} | sorgu {len(sorgu_rows)} (karma %{100*len(sorgu_rows)/len(egitim):.0f})")
print("terim gorevi cikarildi: temel modelin urettigi tanimlar dogrulanmadi")

## 3 egitim

`hucreler/v2/3_egitim.py`

In [ ]:
import inspect, os, torch
from datasets import Dataset
from transformers import Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

SMOKE = False
OUT_DIR = "/content/drive/MyDrive/tus_sft/qwen35_4b_v2" + ("_smoke" if SMOKE else "")
EPOCHS, SEED = 2, 42
SIS_TUS = ("Sen bir tıp uzmanısın. Aşağıdaki TUS sorusunu çöz. "
           "İlk satıra 'Doğru cevap: <harf>) <şık metni>' yaz, ardından kısa bir açıklama ver.")

gpu = torch.cuda.get_device_name(0)
BS, GA = (8, 2) if ("A100" in gpu or "H100" in gpu) else (4, 4)
print(f"{gpu} | batch {BS}x{GA}")

END = "<|im_end|>"
if not hasattr(model, "peft_config"):
    model = FastLanguageModel.get_peft_model(
        model, r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        use_gradient_checkpointing="unsloth", random_state=SEED)
model.print_trainable_parameters()

def encode(row):
    sis = row.get("sistem") or SIS_TUS
    p = tok(prompt(sis, row["question"]), add_special_tokens=False)["input_ids"]
    c = tok(row["answer"] + END, add_special_tokens=False)["input_ids"]
    return {"input_ids": (p+c)[:MAX_SEQ_LEN],
            "labels": ([-100]*len(p) + c)[:MAX_SEQ_LEN],
            "uzun": len(p) >= MAX_SEQ_LEN}

def to_ds(rows):
    ds = Dataset.from_list([{k: r.get(k, "") for k in ("gorev","sistem","question","answer")} for r in rows])
    ds = ds.map(encode, remove_columns=["gorev","sistem","question","answer"])
    ds = ds.filter(lambda x: not x["uzun"]).remove_columns(["uzun"])
    return ds.map(lambda x: {"length": len(x["input_ids"])})

train_ds, val_ds = to_ds(train_rows), to_ds(val_rows)
lens = sorted(train_ds["length"])
print(f"train {len(train_ds)} | val {len(val_ds)} | token p50 {lens[len(lens)//2]} p95 {lens[int(len(lens)*.95)]}")

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = lambda s, v: s + [v]*(n-len(s))
    return {"input_ids":      torch.tensor([pad(b["input_ids"], tok.pad_token_id) for b in batch]),
            "attention_mask": torch.tensor([pad([1]*len(b["input_ids"]), 0)       for b in batch]),
            "labels":         torch.tensor([pad(b["labels"], -100)                for b in batch])}

adim = 20 if SMOKE else 200
_k = set(inspect.signature(TrainingArguments.__init__).parameters)
_ek = {}
if "group_by_length" in _k: _ek["group_by_length"] = True
elif "train_sampling_strategy" in _k: _ek["train_sampling_strategy"] = "group_by_length"
if "warmup_ratio" in _k: _ek["warmup_ratio"] = 0.03
_ek["eval_strategy" if "eval_strategy" in _k else "evaluation_strategy"] = "steps"

args = TrainingArguments(
    output_dir=OUT_DIR, per_device_train_batch_size=BS, per_device_eval_batch_size=BS,
    gradient_accumulation_steps=GA, num_train_epochs=EPOCHS, max_steps=40 if SMOKE else -1,
    learning_rate=1e-4, lr_scheduler_type="cosine", weight_decay=0.0,
    bf16=True, optim="adamw_8bit", logging_steps=10, eval_steps=adim,
    save_strategy="steps", save_steps=adim, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    remove_unused_columns=False, report_to="none", seed=SEED, **_ek)

FastLanguageModel.for_training(model)
tok.padding_side = "right"
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=val_ds, data_collator=collate)
son = get_last_checkpoint(OUT_DIR) if os.path.isdir(OUT_DIR) else None
print("checkpoint:", son)
trainer.train(resume_from_checkpoint=son)
model.save_pretrained(f"{OUT_DIR}/final_adapter")
tokenizer.save_pretrained(f"{OUT_DIR}/final_adapter")
print("kaydedildi:", f"{OUT_DIR}/final_adapter")

## 4 olcum

`hucreler/v2/4_olcum.py`

In [ ]:
import json, re
from math import comb

BASLIK = re.compile(r"^\s*Doğru cevap\s*:?\s*\(?([A-E])\)?")
HER = re.compile(r"(?:Doğru\s*)?(?:cevap|yanıt)\s*[:\-]?\s*\(?([A-E])\)?", re.I)
def parse_cevap(t):
    c = re.sub(r"<think>.*?</think>", "", t or "", flags=re.S).strip()
    m = BASLIK.match(c) or HER.search(c) or re.search(r"\b([A-E])\)", c)
    return m.group(1).upper() if m else None

FastLanguageModel.for_inference(model)

print("[1/2] TUS dogrulugu")
t_out = uret([prompt(SIS_TUS, r["question"]) for r in test_rows], 64, bs=16, etiket="[tus] ")
t_pred = [parse_cevap(o) for o in t_out]
t_acc = sum(p == r["letter"] for p, r in zip(t_pred, test_rows)) / len(test_rows)
print(f"TUS dogruluk: {t_acc:.1%}   (v1: 70.9%)")

print("\n[2/2] talimat takibi — arama sorgusu uretme")
q_out = uret([prompt(SIS_SORGU, r["question"]) for r in test_rows], 48, bs=16, etiket="[sorgu] ")
def kullanilabilir(t, soru):
    s = re.sub(r"<think>.*?</think>", "", t or "", flags=re.S).strip().split("\n")[0].strip(' "\'`')
    s = re.sub(r"^(arama|sorgu|anahtar kelimeler)\s*[:\-]\s*", "", s, flags=re.I).strip()
    if not (3 <= len(s) <= 220): return False, s
    if re.match(r"^[A-E][\)\.]", s): return False, s
    if re.search(r"Doğru cevap", s, re.I): return False, s
    if s[:40].lower() in soru[:70].lower(): return False, s
    return True, s
sonuc = [kullanilabilir(o, r["question"]) for o, r in zip(q_out, test_rows)]
iyi = sum(ok for ok, _ in sonuc)
print(f"kullanilabilir sorgu: {iyi}/{len(test_rows)} (%{100*iyi/len(test_rows):.0f})")
print("  v1 SFT: bozuk (hep cevapliyordu) | ham model: 311/316 (%98)")
for ok, s in sonuc[:6]: print(f"  [{'+' if ok else '-'}] {s[:85]}")

print(f"""
{'='*58}
                     SFT v1      SFT v2
TUS dogrulugu         70,9%      {t_acc:>6.1%}
talimat takibi        bozuk      {iyi/len(test_rows):>6.0%}
{'='*58}
Hedef: TUS dusmemis VE talimat takibi %80 ustu""")

OUT = "/content/drive/MyDrive/tus_sft/qwen35_4b_v2"
with open(f"{OUT}/sonuc_v2.json", "w", encoding="utf-8") as f:
    json.dump({"tus_acc": t_acc, "talimat_takibi": iyi/len(test_rows),
               "train_size": len(train_ds), "v1_tus_acc": 0.7089}, f, ensure_ascii=False, indent=2)
print("kaydedildi:", OUT)